# NB 04: Final Benchmark & Go/No-Go Gate

Head-to-head comparison on the test set with all contestants:
- LR (traditional ratios)
- LR (full features)
- XGBoost (full features)
- GBT (raw XBRL)
- FT-Transformer (tuned, scratch)
- FT-Transformer (tuned, best SSL pretrained)

Then evaluate the go/no-go gate with bootstrap confidence intervals.

**Gate criterion:** FT-Transformer beats XGBoost on >=3/5 distress outcomes
by >=0.01 AUROC on the test set.

**Prerequisites:**
1. Run `00_setup_and_preprocessing.ipynb` first (saves artifact to Drive)
2. Run `02_hp_sweep.ipynb` first (saves sweep results)
3. Run `03_ssl_pretraining.ipynb` first (saves SSL results + checkpoints)
4. Runtime -> Change runtime type -> **T4 GPU** (or A100)
5. GitHub PAT stored in Colab Secrets as `GITHUB_PAT`

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
import os

# Retrieve the GitHub token from Colab Secrets
github_token = userdata.get('GITHUB_PAT')

username = "elroy-galbraith"
repository = "fin-jepa"
repo_url = f"https://{github_token}@github.com/{username}/{repository}.git"

if not os.path.exists(f'/content/{repository}'):
    !git clone {repo_url} /content/{repository}

%cd /content/{repository}

# Install fin-jepa as editable WITHOUT upgrading Colab's pre-installed deps
!pip install -q --no-deps -e '.[dev]'

# Install only the deps Colab doesn't already ship
!pip install -q hydra-core omegaconf mlflow optuna xgboost yfinance rich python-dotenv tqdm

In [ ]:
# Symlink data from Google Drive so relative paths work
import os

DRIVE_DATA = '/content/drive/MyDrive/Fin-JEPA/data'

for subdir in ['raw', 'processed']:
    target = f'data/{subdir}'
    if os.path.islink(target) or os.path.isdir(target):
        !rm -rf {target}
    os.symlink(f'{DRIVE_DATA}/{subdir}', target)

# Verify data exists
!ls -la data/raw/xbrl_features.parquet
!ls -la data/processed/label_database.parquet
!ls -la data/raw/market/market_aligned.parquet
print('Data linked successfully!')

# Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU detected! Change runtime to T4 or A100.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
props = torch.cuda.get_device_properties(0)
print(f'VRAM: {props.total_memory / 1e9:.1f} GB')

In [ ]:
# Common setup: logging, configs, control flag
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from omegaconf import OmegaConf

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# Set to True to re-run experiments even if cached results exist
FORCE_RERUN = False

# Load configs from YAML (override individual values below if needed)
benchmark_cfg = OmegaConf.load('configs/study0/benchmark.yaml')
ssl_cfg = OmegaConf.load('configs/study0/ssl_experiment.yaml')

# Paths
RESULTS_DIR = Path('results/study0')
BENCHMARK_PATH = RESULTS_DIR / 'benchmark_results.json'
SWEEP_PATH = RESULTS_DIR / 'ft_sweep_results.json'
SSL_V2_PATH = RESULTS_DIR / 'ssl_experiment_results_v2.json'
FINAL_PATH = RESULTS_DIR / 'final_benchmark.json'

OUTCOMES = ['stock_decline', 'earnings_restate', 'audit_qualification',
            'sec_enforcement', 'bankruptcy']

print(f'Results dir: {RESULTS_DIR}')
print(f'Benchmark cached: {BENCHMARK_PATH.exists()}')
print(f'Sweep cached: {SWEEP_PATH.exists()}')
print(f'SSL v2 cached: {SSL_V2_PATH.exists()}')
print(f'FORCE_RERUN: {FORCE_RERUN}')

# Preprocessing artifact path (shared across split notebooks)
ARTIFACT_DIR = Path('/content/drive/MyDrive/Fin-JEPA/artifacts/study0')
ARTIFACT_PATH = ARTIFACT_DIR / 'preprocessing_v1.pkl'

SEED = int(OmegaConf.select(benchmark_cfg, 'training.seed', default=42))

mask_ratios = [0.15, 0.30, 0.50]
MIN_POSITIVES = 20


In [ ]:
import pickle

if not ARTIFACT_PATH.exists():
    raise FileNotFoundError(
        f'Preprocessing artifact not found at {ARTIFACT_PATH}.\n'
        'Run 00_setup_and_preprocessing.ipynb first.'
    )

with open(ARTIFACT_PATH, 'rb') as f:
    _artifact = pickle.load(f)

splits = _artifact['splits']
scaler = _artifact['scaler']
feature_cols = _artifact['feature_cols']
categorical_cols = _artifact['categorical_cols']
n_features = _artifact['n_features']
n_cat = _artifact['n_cat']
cat_cards = _artifact['cat_cards']
config_fingerprint = _artifact['config_fingerprint']

print(f'Loaded preprocessing artifact (fingerprint: {config_fingerprint})')
print(f'  Train: {len(splits["train"]):,} | Val: {len(splits["val"]):,} | Test: {len(splits["test"]):,}')
print(f'  Features: {n_features} ({n_cat} categorical)')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Load sweep results
SWEEP_PATH = RESULTS_DIR / 'ft_sweep_results.json'
if not SWEEP_PATH.exists():
    raise FileNotFoundError('Sweep results not found. Run 02_hp_sweep.ipynb first.')
with open(SWEEP_PATH) as f:
    sweep_results = json.load(f)
best_cfg = sweep_results['best_config']

# Load SSL results
SSL_V2_PATH = RESULTS_DIR / 'ssl_experiment_results_v2.json'
if not SSL_V2_PATH.exists():
    raise FileNotFoundError('SSL results not found. Run 03_ssl_pretraining.ipynb first.')
with open(SSL_V2_PATH) as f:
    ssl_results = json.load(f)

print(f"Loaded sweep (best: d_token={best_cfg['d_token']}, n_layers={best_cfg['n_layers']})")
print(f"Loaded SSL v2 results ({len(ssl_results.get('pretrained', {}))} mask ratios)")

In [ ]:
# Load baseline benchmark results (from NB 01)
BENCHMARK_PATH = RESULTS_DIR / 'benchmark_results.json'
if BENCHMARK_PATH.exists():
    with open(BENCHMARK_PATH) as f:
        benchmark_results = json.load(f)
    if 'outcomes' in benchmark_results:
        benchmark_outcomes = benchmark_results['outcomes']
    else:
        benchmark_outcomes = benchmark_results
    print(f'Loaded baseline results: {list(benchmark_outcomes.keys())}')
else:
    benchmark_outcomes = {}
    print('WARNING: baseline results not found -- some comparison cells may be incomplete')

# highlight_best helper (needed by comparison tables)
def highlight_best(s, metric):
    """Bold-green the best value per row; lower is better for Brier/ECE."""
    better_low = metric in ('brier', 'ece')
    valid = s.dropna()
    if valid.empty:
        return ['' for _ in s]
    best_val = valid.min() if better_low else valid.max()
    return ['font-weight: bold; background-color: #d4edda' if v == best_val else '' for v in s]

mask_ratios = sorted(ssl_results.get('pretrained', {}).keys())

## 5. Final Comparison: All Models

Head-to-head on the test set with all contestants:
- LR (traditional ratios)
- LR (full features)
- XGBoost (full features)
- GBT (raw XBRL)
- FT-Transformer (tuned, scratch)
- FT-Transformer (tuned, best SSL pretrained)

In [ ]:
# Unified 6-model comparison — one table per metric, best cell highlighted
ALL_MODEL_KEYS = ['XGBoost', 'LR (full)', 'LR (trad)', 'GBT (raw)', 'FT tuned', 'FT tuned+SSL']
best_sweep = max(sweep_results['configs'], key=lambda c: c['mean_auroc'])

def get_combined_metrics():
    out = {model: {} for model in ALL_MODEL_KEYS}
    for outcome in OUTCOMES:
        bm = benchmark_outcomes.get(outcome, {})
        for src_key, disp in [('xgboost', 'XGBoost'), ('lr_full', 'LR (full)'),
                               ('lr_traditional', 'LR (trad)'), ('gbt_raw', 'GBT (raw)')]:
            rec = bm.get(src_key) or {}
            out[disp][outcome] = {m: rec.get(m) for m in ['auroc', 'auprc', 'brier', 'ece']}
        rec_ft = (best_sweep.get('outcomes') or {}).get(outcome) or {}
        out['FT tuned'][outcome] = {m: rec_ft.get(m) for m in ['auroc', 'auprc', 'brier', 'ece']}
        best_ssl_rec, best_ssl_auroc = {}, -1
        for mr in mask_ratios:
            rec = ((ssl_results.get('pretrained') or {}).get(mr) or {}).get(outcome) or {}
            if rec.get('auroc') is not None and rec['auroc'] > best_ssl_auroc:
                best_ssl_auroc = rec['auroc']
                best_ssl_rec = rec
        out['FT tuned+SSL'][outcome] = {m: best_ssl_rec.get(m) for m in ['auroc', 'auprc', 'brier', 'ece']}
    return out

combined_metrics = get_combined_metrics()

for metric in ['auroc', 'auprc', 'brier', 'ece']:
    rows = []
    for outcome in OUTCOMES:
        row = {'Outcome': outcome}
        for model in ALL_MODEL_KEYS:
            row[model] = (combined_metrics[model].get(outcome) or {}).get(metric)
        rows.append(row)
    df = pd.DataFrame(rows).set_index('Outcome')
    print(f'\n=== {metric.upper()} ===')
    display(df.style.apply(highlight_best, metric=metric, axis=1).format('{:.4f}', na_rep='--'))


In [ ]:
# Hero chart: all models compared
# Construct combined_df for AUROC from combined_metrics
import pandas as pd

plot_rows = []
for outcome in OUTCOMES:
    row = {'Outcome': outcome}
    for model in ALL_MODEL_KEYS:
        row[model] = (combined_metrics[model].get(outcome) or {}).get('auroc')
    plot_rows.append(row)

combined_df = pd.DataFrame(plot_rows).set_index('Outcome')

plot_cols = [c for c in ALL_MODEL_KEYS if c in combined_df.columns]
plot_data = combined_df[plot_cols].dropna(how='all')

if not plot_data.empty:
    colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f', '#edc948']
    ax = plot_data.plot(kind='bar', figsize=(14, 6), width=0.85,
                        color=colors[:len(plot_cols)], edgecolor='white')
    ax.set_ylabel('Test AUROC')
    ax.set_title('Study 0 Final Benchmark: All Models (Test Set)')
    ax.set_ylim(0.5, max(plot_data.max().max() + 0.05, 0.8))
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.legend(loc='upper right', fontsize=9)
    import matplotlib.pyplot as plt
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    fig_dir = RESULTS_DIR / 'figures'
    fig_dir.mkdir(exist_ok=True)
    plt.savefig(fig_dir / 'hero_chart_all_models.png', bbox_inches='tight', dpi=300)
    plt.show()
else:
    print('No combined results to plot.')


In [ ]:
# Bootstrap CI — canonical implementation from fin_jepa.training.metrics
from fin_jepa.training.metrics import bootstrap_auroc_ci

def bootstrap_auroc_delta(y_true_ft, y_score_ft, y_true_xgb, y_score_xgb,
                          n_bootstrap=1000, seed=42):
    """Thin backward-compat wrapper around bootstrap_auroc_ci.
    Returns (mean_delta, ci_low, ci_high).
    y_true_ft and y_true_xgb must be the same array (paired test set).
    """
    result = bootstrap_auroc_ci(
        y_true_ft, y_score_ft, y_score_xgb,
        n_bootstrap=n_bootstrap, seed=seed,
    )
    return result['estimate'], result['ci_lower'], result['ci_upper']


In [ ]:
%%time
# Re-generate test predictions: all 6 models x all outcomes x 4 metrics + paired bootstrap CIs
FINAL_RESULT_PATH = RESULTS_DIR / 'final_benchmark.json'

if not FORCE_RERUN and FINAL_RESULT_PATH.exists():
    print(f'Loading cached final benchmark from {FINAL_RESULT_PATH}')
    with open(FINAL_RESULT_PATH) as f:
        final_results = json.load(f)
else:
    import xgboost as xgb
    from fin_jepa.training.metrics import compute_all_metrics
    from fin_jepa.models.baselines import build_logistic_regression, build_gbt
    from fin_jepa.data.feature_engineering import TRADITIONAL_RATIO_FEATURES, RAW_FEATURES
    from fin_jepa.training.ablations import _train_and_evaluate

    best = sweep_results['best_config']
    model_kwargs = {
        'n_features': n_features,
        'd_token': best['d_token'],
        'n_heads': best['n_heads'],
        'n_layers': best['n_layers'],
        'd_ffn_factor': 4,
        'dropout': 0.0,
        'n_outputs': 1,
        'n_cat_features': n_cat,
        'cat_cardinalities': cat_cards,
    }

    # Identify best SSL checkpoint
    ssl_ckpt_dir = Path('models/checkpoints/study0_ssl_experiment_v2')
    best_mr_key, best_mr_auroc = None, -1
    for mr in mask_ratios:
        aurocs = [ssl_results['pretrained'].get(mr, {}).get(o, {}).get('auroc', 0) for o in OUTCOMES]
        avg = np.mean([a for a in aurocs if a and a > 0]) if any(a and a > 0 for a in aurocs) else 0
        if avg > best_mr_auroc:
            best_mr_auroc = avg
            best_mr_key = mr
    ssl_ckpt_path = ssl_ckpt_dir / f'encoder_mr{best_mr_key}.pt'
    ssl_state_dict = torch.load(ssl_ckpt_path, map_location=device) if ssl_ckpt_path.exists() else None

    _MODEL_KEYS = ['lr_trad', 'lr_full', 'xgboost', 'gbt_raw', 'ft_scratch', 'ft_ssl']
    final_results = {k: {} for k in _MODEL_KEYS}
    # Keep gate_* keys for backward compat with gate-evaluation cells
    final_results.update({'gate_scratch': {}, 'gate_ssl': {}, 'gate_xgb': {}, 'bootstrap': {}})
    batch_size = 256
    _cat_cols = categorical_cols or None
    trad_cols = [c for c in TRADITIONAL_RATIO_FEATURES if c in feature_cols]
    raw_cols  = [c for c in feature_cols if c in RAW_FEATURES]

    for outcome in OUTCOMES:
        test_df  = splits['test'][splits['test'][outcome].notna()]
        train_df = splits['train'][splits['train'][outcome].notna()]
        val_df   = splits['val'][splits['val'][outcome].notna()]
        n_pos = int(train_df[outcome].sum())
        if n_pos < MIN_POSITIVES:
            print(f'{outcome}: skipped (< {MIN_POSITIVES} positives)')
            continue
        n_neg = len(train_df) - n_pos
        pos_weight = n_neg / max(n_pos, 1)

        X_train = np.nan_to_num(train_df[feature_cols].values.astype(np.float32), nan=0.0)
        X_test  = np.nan_to_num(test_df[feature_cols].values.astype(np.float32),  nan=0.0)
        y_train = train_df[outcome].values.astype(np.float32)
        y_test  = test_df[outcome].values.astype(np.float32)

        # LR (full features)
        lr_full_m = build_logistic_regression(C=1.0, max_iter=1000)
        lr_full_m.fit(X_train, y_train)
        final_results['lr_full'][outcome] = compute_all_metrics(
            y_test, lr_full_m.predict_proba(X_test)[:, 1])

        # LR (traditional ratios)
        if trad_cols:
            trad_idx = [feature_cols.index(c) for c in trad_cols]
            lr_trad_m = build_logistic_regression(C=1.0, max_iter=1000)
            lr_trad_m.fit(X_train[:, trad_idx], y_train)
            final_results['lr_trad'][outcome] = compute_all_metrics(
                y_test, lr_trad_m.predict_proba(X_test[:, trad_idx])[:, 1])

        # XGBoost
        xgb_model = xgb.XGBClassifier(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight,
            eval_metric='auc', random_state=42, tree_method='hist',
        )
        xgb_model.fit(X_train, y_train)
        xgb_scores = xgb_model.predict_proba(X_test)[:, 1]
        xgb_m = compute_all_metrics(y_test, xgb_scores)
        final_results['xgboost'][outcome] = xgb_m
        final_results['gate_xgb'][outcome] = {'auroc': xgb_m['auroc']}

        # GBT (raw XBRL)
        if raw_cols:
            raw_idx = [feature_cols.index(c) for c in raw_cols]
            gbt_m = build_gbt(max_iter=500, learning_rate=0.05, max_depth=6, min_samples_leaf=20)
            gbt_m.fit(X_train[:, raw_idx], y_train)
            final_results['gbt_raw'][outcome] = compute_all_metrics(
                y_test, gbt_m.predict_proba(X_test[:, raw_idx])[:, 1])

        # ATS-217: use _train_and_evaluate for FT-Transformer — the exact
        # same function used by run_ssl_experiment and run_multiseed_benchmark.
        # This guarantees seed-42 numbers are identical across all result files.

        # FT-Transformer (scratch, tuned)
        seed_everything(SEED)
        scratch_m = _train_and_evaluate(
            splits, feature_cols, outcome, device, model_kwargs,
            cat_feature_cols=categorical_cols, lr=best['lr'],
            seed=SEED,
        )
        final_results['ft_scratch'][outcome] = scratch_m
        final_results['gate_scratch'][outcome] = {'auroc': scratch_m['auroc']}

        # FT-Transformer (SSL pretrained)
        if ssl_state_dict is not None:
            seed_everything(SEED)
            ssl_m = _train_and_evaluate(
                splits, feature_cols, outcome, device, model_kwargs,
                init_state_dict=ssl_state_dict,
                cat_feature_cols=categorical_cols, lr=best['lr'],
                seed=SEED,
            )
            final_results['ft_ssl'][outcome] = ssl_m
            final_results['gate_ssl'][outcome] = {'auroc': ssl_m['auroc']}
        else:
            ssl_m = scratch_m

        # Bootstrap CIs — re-train to get raw predictions for paired test
        seed_everything(SEED)
        train_loader = make_dataloader(train_df, feature_cols, outcome, batch_size,
                                       shuffle=True, cat_feature_cols=_cat_cols, seed=SEED)
        val_loader = make_dataloader(val_df, feature_cols, outcome, batch_size,
                                     shuffle=False, cat_feature_cols=_cat_cols)
        test_loader = make_dataloader(test_df, feature_cols, outcome, batch_size,
                                      shuffle=False, cat_feature_cols=_cat_cols)

        m_scratch = FTTransformer(**model_kwargs).to(device)
        opt = torch.optim.AdamW(m_scratch.parameters(), lr=best['lr'], weight_decay=1e-5)
        crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
        train_ft_transformer(m_scratch, train_loader, val_loader, crit, opt,
                             device, epochs=100, patience=10)
        yt_scratch, ys_scratch = _predict_scores(m_scratch, test_loader, device)
        ci_scratch = bootstrap_auroc_delta(yt_scratch, ys_scratch, y_test, xgb_scores)

        if ssl_state_dict is not None:
            seed_everything(SEED)
            m_ssl_b = FTTransformer(**model_kwargs).to(device)
            m_ssl_b.load_state_dict(ssl_state_dict, strict=False)
            opt_s = torch.optim.AdamW(m_ssl_b.parameters(), lr=best['lr'], weight_decay=1e-5)
            crit_s = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
            train_ft_transformer(m_ssl_b, train_loader, val_loader, crit_s, opt_s,
                                 device, epochs=100, patience=10)
            yt_ssl, ys_ssl = _predict_scores(m_ssl_b, test_loader, device)
            ci_ssl = bootstrap_auroc_delta(yt_ssl, ys_ssl, y_test, xgb_scores)
        else:
            ci_ssl = (None, None, None)

        final_results['bootstrap'][outcome] = {
            'scratch_vs_xgb': {'mean': ci_scratch[0], 'ci_low': ci_scratch[1], 'ci_high': ci_scratch[2]},
            'ssl_vs_xgb':     {'mean': ci_ssl[0],     'ci_low': ci_ssl[1],     'ci_high': ci_ssl[2]},
        }
        best_m = ssl_m if ssl_state_dict else scratch_m
        print(f"{outcome}: FT_ssl={best_m.get('auroc', 0):.4f}  "
              f"XGB={xgb_m['auroc']:.4f}  "
              f"delta={best_m.get('auroc', 0) - xgb_m['auroc']:+.4f}")

    import hashlib as _hl
    _fp_dict = {'d_token': best['d_token'], 'n_heads': best['n_heads'],
                'n_layers': best['n_layers'], 'lr': best['lr']}
    _fp = _hl.sha256(json.dumps(_fp_dict, sort_keys=True).encode()).hexdigest()[:16]
    final_results['_meta'] = {
        'spec': 'ATS-176',
        'generated_at': pd.Timestamp.now().isoformat(),
        'models': _MODEL_KEYS,
        'metrics': ['auroc', 'auprc', 'brier', 'ece'],
        'ci_method': 'paired_bootstrap',
        'n_bootstrap': 1000,
        'best_ssl_mr': best_mr_key,
        'config_fingerprint': _fp,
    }
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(FINAL_RESULT_PATH, 'w') as f:
        json.dump(final_results, f, indent=2, default=str)
    print(f'\nFinal benchmark saved to {FINAL_RESULT_PATH}')


## 6. Go/No-Go Gate with Bootstrap CIs

**Rule:** FT-Transformer must beat XGBoost on â‰¥3 of 5 outcomes by â‰¥0.01 AUROC on the test set.

We evaluate with:
1. FT-Transformer (tuned, scratch) vs XGBoost
2. FT-Transformer (tuned, best SSL) vs XGBoost

Bootstrap confidence intervals (1000 resamples) on AUROC differences.

In [ ]:
from fin_jepa.training.metrics import go_no_go_gate

In [ ]:
# Go/No-Go Gate
print('=' * 65)
print('GO/NO-GO GATE: FT-Transformer (tuned) vs XGBoost')
print('Rule: FT must beat XGB on >= 3/5 outcomes by >= 0.01 AUROC')
print('=' * 65)

# Use final_results for gate (re-trained with tuned hyperparams)
xgb_gate = final_results.get('gate_xgb', {})
ft_scratch_gate = final_results.get('gate_scratch', {})
ft_ssl_gate = final_results.get('gate_ssl', {})
bootstrap = final_results.get('bootstrap', {})

gate_outcomes_s = [o for o in OUTCOMES if o in xgb_gate and o in ft_scratch_gate
                   and ft_scratch_gate[o].get('auroc') is not None]
gate_outcomes_ssl = [o for o in OUTCOMES if o in xgb_gate and o in ft_ssl_gate
                     and ft_ssl_gate[o].get('auroc') is not None]

print('\n--- Gate 1: FT-Transformer (tuned, scratch) vs XGBoost ---')
if gate_outcomes_s:
    passed_s, wins_s, detail_s = go_no_go_gate(ft_scratch_gate, xgb_gate, gate_outcomes_s)
    status_s = 'PASSED' if passed_s else 'FAILED'
    print(f'Result: {status_s} ({wins_s}/{len(gate_outcomes_s)} wins)\n')
    for o, d in detail_s.items():
        marker = '+' if d['win'] else '-'
        delta = d['ft_auroc'] - d['xgb_auroc']
        bs = bootstrap.get(o, {}).get('scratch_vs_xgb', {})
        ci_str = f"  95%CI=[{bs['ci_low']:+.4f}, {bs['ci_high']:+.4f}]" if bs else ''
        print(f'  [{marker}] {o:25s}  FT={d["ft_auroc"]:.4f}  XGB={d["xgb_auroc"]:.4f}  '
              f'delta={delta:+.4f}{ci_str}')
else:
    print('No overlapping outcomes to evaluate.')

print('\n--- Gate 2: FT-Transformer (tuned, best SSL) vs XGBoost ---')
if gate_outcomes_ssl:
    passed_ssl, wins_ssl, detail_ssl = go_no_go_gate(ft_ssl_gate, xgb_gate, gate_outcomes_ssl)
    status_ssl = 'PASSED' if passed_ssl else 'FAILED'
    print(f'Result: {status_ssl} ({wins_ssl}/{len(gate_outcomes_ssl)} wins)\n')
    for o, d in detail_ssl.items():
        marker = '+' if d['win'] else '-'
        delta = d['ft_auroc'] - d['xgb_auroc']
        bs = bootstrap.get(o, {}).get('ssl_vs_xgb', {})
        ci_str = f"  95%CI=[{bs['ci_low']:+.4f}, {bs['ci_high']:+.4f}]" if bs else ''
        print(f'  [{marker}] {o:25s}  FT={d["ft_auroc"]:.4f}  XGB={d["xgb_auroc"]:.4f}  '
              f'delta={delta:+.4f}{ci_str}')
else:
    print('No overlapping outcomes to evaluate.')

In [ ]:
# Bootstrap CI forest plot
bs_data = final_results.get('bootstrap', {})
plot_outcomes = [o for o in OUTCOMES if o in bs_data and 'scratch_vs_xgb' in bs_data[o]]

if plot_outcomes:
    fig, ax = plt.subplots(figsize=(10, len(plot_outcomes) * 0.8 + 2))
    y_pos = np.arange(len(plot_outcomes))
    width = 0.35

    for i, outcome in enumerate(plot_outcomes):
        # Scratch vs XGB
        bs_s = bs_data[outcome]['scratch_vs_xgb']
        ax.errorbar(bs_s['mean'], i - width/2,
                    xerr=[[bs_s['mean'] - bs_s['ci_low']], [bs_s['ci_high'] - bs_s['mean']]],
                    fmt='o', color='#59a14f', capsize=5, markersize=8,
                    label='FT tuned (scratch)' if i == 0 else '')

        # SSL vs XGB
        if 'ssl_vs_xgb' in bs_data[outcome]:
            bs_ssl = bs_data[outcome]['ssl_vs_xgb']
            ax.errorbar(bs_ssl['mean'], i + width/2,
                        xerr=[[bs_ssl['mean'] - bs_ssl['ci_low']], [bs_ssl['ci_high'] - bs_ssl['mean']]],
                        fmt='s', color='#edc948', capsize=5, markersize=8,
                        label='FT tuned+SSL' if i == 0 else '')

    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.7)
    ax.axvline(x=0.01, color='green', linestyle=':', alpha=0.5, label='Gate margin (0.01)')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_outcomes)
    ax.set_xlabel('AUROC Delta (FT - XGBoost)')
    ax.set_title('Bootstrap 95% CI: FT-Transformer vs XGBoost AUROC Difference')
    ax.legend(loc='lower right')
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    fig_dir = RESULTS_DIR / 'figures'
    fig_dir.mkdir(exist_ok=True)
    plt.savefig(fig_dir / 'bootstrap_ci_forest_plot.png', bbox_inches='tight', dpi=300)
    plt.show()
else:
    print('No bootstrap data to plot.')
